# Mission 05: Multi-layered Prompt & Cache - 해답 노트북

이 노트북은 다섯 번째 미션의 완성된 솔루션 코드와 설명입니다.

In [ ]:
# 1. 환경 준비
import sys
import os
from dotenv import load_dotenv

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("app"))
load_dotenv(override=True)

from app.utils.llm import get_llm
from langchain_core.messages import SystemMessage, HumanMessage

### [미션 1] 계층형 PromptManager 구현하기

아래 코드는 L1 ~ L4의 계층별 프롬프트 블록을 합쳐 시스템 프롬프트를 렌더링하고 동적 데이터를 치환하는 PromptManager 솔루션 코드입니다.

In [ ]:
class PromptManager:
    def __init__(self):
        # L1: 핵심 페르소나 정의
        self.l1_role = "You are Antigravity, a professional production-grade software engineering agent."
        
        # L2: 운영 가이드라인
        self.l2_guidelines = (
            "1. Always verify code execution before reporting completion.\n"
            "2. Follow secure coding standards to prevent injection and exposure vulnerabilities.\n"
            "3. Maintain structured logs for all file reads and writes."
        )
        
        # L3: 도구 스펙 (Dynamic Tools Spec)
        self.l3_tools = "Available tools are registered dynamically at the runtime loop."
        
        # L4: 대규모 컨텍스트 (정적 레퍼런스 데이터)
        self.l4_context = ""
        
    def set_reference_context(self, context_text: str):
        self.l4_context = context_text
        
    def build_system_prompt(self, dynamic_state: dict) -> str:
        """
        L1 ~ L4 레이어를 결합하여 에이전트에게 공급할 완성형 System Prompt를 생성합니다.
        dynamic_state 사전을 통해 런타임 정보(예: 현재 사용자 권한, 세션 날짜 등)를 주입받아 포맷팅합니다.
        """
        permission_string = f"Current User Permissions: {dynamic_state.get('user_permission', 'NONE')}"
        project_string = f"Active Target Project: {dynamic_state.get('active_project', 'NONE')}"
        
        guidelines_with_state = f"{self.l2_guidelines}\n- {permission_string}\n- {project_string}"
        
        full_prompt = (
            f"=== ROLE (L1) ===\n{self.l1_role}\n\n"
            f"=== OPERATING GUIDELINES (L2) ===\n{guidelines_with_state}\n\n"
            f"=== TOOLS SPECS (L3) ===\n{self.l3_tools}\n\n"
            f"=== STATIC REFERENCE CONTEXT (L4) ===\n{self.l4_context}"
        )
        return full_prompt

pm = PromptManager()
print("PromptManager 객체 초기화 성공!")

### [미션 2] 프롬프트 렌더링 및 캐시 지정 시뮬레이션

대용량 더미 문서 데이터를 L4 영역에 설정하고, `PromptManager`를 통해 시스템 프롬프트를 구성해 봅니다.

In [ ]:
# 대규모 정적 API 레퍼런스 문서 모사 (캐싱 대상)
large_api_docs = """=== API Reference Manual ===\n""" + "\n".join(
    [f"Function_ID_{i}: Perform operation {i}. Parameters: arg{i}. Returns result." for i in range(500)]
)

pm.set_reference_context(large_api_docs)

state = {
    "user_permission": "READ_WRITE_EXECUTE",
    "active_project": "harness_agent_lab"
}

system_prompt = pm.build_system_prompt(state)

print(f"생성된 시스템 프롬프트 크기: {len(system_prompt)} 글자")
print("상단 300글자 요약:\n", system_prompt[:300])

### [미션 3] 에이전트 캐싱 지연시간 최적화 검증

캐싱이 걸렸을 때의 Latency 단축 효과를 확인하기 위해 시뮬레이션 테스트를 수행합니다.

In [ ]:
import time
llm = get_llm(model_name="google_vertexai:gemini-3.5-flash", temperature=0.0)

print("🔄 1회차 호출 (Cold Start - 캐시 생성) 시작...")
t1 = time.time()
res1 = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="Function_ID_256번 API의 매개변수와 반환 스펙이 무엇인지 설명해줘.")
])
cold_latency = time.time() - t1
print(f"✅ 1회차 호출 완료! (소요 시간: {cold_latency:.2f}초)")
print("답변:", res1.content)

print("\n" + "-"*50 + "\n")

print("🔄 2회차 호출 (Warm Start - 캐시 히트) 시작...")
t2 = time.time()
res2 = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="Function_ID_128번 API의 매개변수와 반환 스펙은 뭐야?")
])
warm_latency = time.time() - t2
print(f"✅ 2회차 호출 완료! (소요 시간: {warm_latency:.2f}초)")
print("답변:", res2.content)

speedup = cold_latency / warm_latency if warm_latency > 0 else 1.0
print(f"\n🚀 캐싱 적용으로 속도가 약 {speedup:.1f}배 개선되었습니다.")